In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

# 1. 定义数据
method_list = [
    "SD-3.5-M",
    "AFPO (HPDv3)",
    "AFPO (Civitai-top)",
    "FlowGRPO",
    "FG+AFPO (HPDv3)", # [新增] 新方法
    "FG+AFPO (Civitai-top)", 
    "GRPO-Guard",
    "DiffusionNFT",
]

human_preference_specific_dict = {
    "SD-3.5-M": { "UniRwd": 3.30, "HPSv3": 10.03 },
    "FlowGRPO": { "UniRwd": 3.59, "HPSv3": 12.63 },
    "FG+AFPO (HPDv3)": { "UniRwd": 3.54, "HPSv3": 13.29 }, # [新增] 数据
    "FG+AFPO (Civitai-top)": { "UniRwd": 3.62, "HPSv3": 12.90 }, 
    "GRPO-Guard": { "UniRwd": 3.64, "HPSv3": 12.51 },
    "DiffusionNFT": { "UniRwd": 3.68, "HPSv3": 12.47 },
    "AFPO (HPDv3)": { "UniRwd": 3.46, "HPSv3": 12.77 },
    "AFPO (Civitai-top)": { "UniRwd": 3.57, "HPSv3": 12.88 }
}

oneig_benchmark_dict = {
    "SD-3.5-M": 0.357,
    "FlowGRPO": 0.260,
    "FG+AFPO (HPDv3)": 0.305, # [新增] 数据
    "FG+AFPO (Civitai-top)": 0.350, 
    "GRPO-Guard": 0.241,
    "DiffusionNFT": 0.175,
    "AFPO (HPDv3)": 0.321,
    "AFPO (Civitai-top)": 0.334,
}

# 2. 归一化处理
uni_rwd_values = [human_preference_specific_dict[m]["UniRwd"] for m in method_list]
hpsv3_values = [human_preference_specific_dict[m]["HPSv3"] for m in method_list]

uni_rwd_min, uni_rwd_max = min(uni_rwd_values), max(uni_rwd_values)
hpsv3_min, hpsv3_max = min(hpsv3_values), max(hpsv3_values)

def normalize(value, min_val, max_val):
    if max_val == min_val: return 0.0
    return (value - min_val) / (max_val - min_val)

weight_uni_rwd = 0.5
weight_hpsv3 = 0.5

human_preference_dict = {}
for m in method_list:
    uni_rwd_norm = normalize(human_preference_specific_dict[m]["UniRwd"], uni_rwd_min, uni_rwd_max)
    hpsv3_norm = normalize(human_preference_specific_dict[m]["HPSv3"], hpsv3_min, hpsv3_max)
    weighted_avg = weight_uni_rwd * uni_rwd_norm + weight_hpsv3 * hpsv3_norm
    human_preference_dict[m] = weighted_avg

# 3. 绘图设置
plt.rcParams.update({
    "axes.labelsize": 14,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "legend.fontsize": 8,
})

fig, ax = plt.subplots(figsize=(5, 4), dpi=300)

# 更新 markers 和 colors 列表以匹配 method_list 的长度 (8个元素)
# 对应索引: 0:SD, 1:AFPO(H), 2:AFPO(C)-Skip, 3:Flow, 4:New(HPD)-Skip, 5:New(Civ)-Skip, 6:GRPO, 7:NFT
markers_default = ["o", "s", "skip", "D", "skip", "skip", "P", "X"]
colors_default = ["#A4C6E2", "#E4A39D", "skip", "#AFD498", "skip", "skip", "#AE9DC7", "#e7dbd3"]

# Plot points
for i, m in enumerate(method_list):
    x = human_preference_dict[m]
    y = oneig_benchmark_dict[m]

    # --- 样式逻辑 ---
    if m == "AFPO (Civitai-top)":
        mk = "*"
        c = "#EF9963"
        s = 320
        zorder = 10
        lw = 1.0
        alpha = 1.0
    
    # [新增] 针对 FG+AFPO (Civitai-top) 的特定样式
    elif m == "FG+AFPO (Civitai-top)":
        mk = "p"          # 五边形 (Pentagon) - 为了区分
        c = "#FFEEBA"     # 淡黄色 (Light Yellow)
        s = 160 * 1.4           # 稍小一点点
        zorder = 11       # 最顶层
        lw = 1.0
        alpha = 1.0

    # [新增] 针对 FG+AFPO (HPDv3) 的特定样式
    elif m == "FG+AFPO (HPDv3)":
        mk = "h"          # 六边形 (Hexagon)
        c = "#D86983"     # 青绿色 (Turquoise) "#76D7C4"
        s = 160 * 1.3           # 160 * 1.4 = 224
        zorder = 11       # 最顶层
        lw = 1.0
        alpha = 0.95
        
    else:
        mk = markers_default[i]
        c = colors_default[i]
        zorder = 3
        lw = 0.8
        alpha = 1.0
        s = 160
        
        if m == "FlowGRPO":
            s = 96
        elif m == "AFPO (HPDv3)":
            s = 120
            alpha = 0.8

    ax.scatter(
        x, y,
        s=s, marker=mk, facecolors=c, edgecolors="black",
        linewidths=lw, zorder=zorder, label=m, alpha=alpha
    )

# 4. 标签布局
label_offsets = {
    "SD-3.5-M": (10, 0),
    "FlowGRPO": (-8, 5),            
    "GRPO-Guard": (10, -2),
    "DiffusionNFT": (-8, 5),
    "AFPO (HPDv3)": (-10, 0),
    "AFPO (Civitai-top)": (8, -8),
    "FG+AFPO (Civitai-top)": (0, 14),
    "FG+AFPO (HPDv3)": (8, -10) 
}

fontsize_label = 9
for m in method_list:
    x = human_preference_dict[m]
    y = oneig_benchmark_dict[m]
    dx, dy = label_offsets.get(m, (5, 5))
    
    if dx < 0: ha = 'right'
    elif dx > 0: ha = 'left'
    else: ha = 'center'
    
    font_weight = "normal"
    
    ax.annotate(
        m, (x, y),
        xytext=(dx, dy), textcoords="offset points",
        fontsize=fontsize_label, color="#333333", fontweight=font_weight,
        ha=ha, va='center', zorder=20
    )

# Axes settings
ax.set_xlabel(r"Human Preference$\uparrow$")
ax.set_ylabel(r"Stylization$\uparrow$")
ax.grid(True, linestyle="--", linewidth=0.8, alpha=0.3, zorder=0)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

# Ticks
ax.set_xlim(-0.1, 1.1)
ax.set_xticks(np.arange(0.0, 1.1, 0.2))
ax.set_ylim(0.14, 0.41)
ax.set_yticks(np.arange(0.15, 0.41, 0.05))
ax.xaxis.set_major_formatter(FuncFormatter(lambda v, pos: f"{v:.1f}"))
ax.yaxis.set_major_formatter(FuncFormatter(lambda v, pos: f"{v:.2f}"))

# Legend
legend = ax.legend(
    loc="lower left", frameon=True, fancybox=True, edgecolor="gray",
    framealpha=0.3, borderpad=0.5, labelspacing=0.8,
    handletextpad=0.5, borderaxespad=0.8
)

# Legend marker sizing
# 注意：这里改用了 legendHandles 以兼容新版 matplotlib
for handle in legend.legend_handles:
    lbl = handle.get_label()
    if lbl == "AFPO (Civitai-top)":
        handle.set_sizes([180.0]) 
    elif lbl == "FG+AFPO (Civitai-top)":
        handle.set_sizes([140.0])
    elif lbl == "FG+AFPO (HPDv3)": # [新增] 图例大小
        handle.set_sizes([100*1.3])
    elif lbl == "AFPO (HPDv3)":
        handle.set_sizes([80.0]) 
    elif lbl == "FlowGRPO":
        handle.set_sizes([75.0]) 
    else:
        handle.set_sizes([100.0]) 

fig.tight_layout()
fig.savefig("2026-ICML-human_preference-vs-oneig_style-v2.pdf", dpi=300, bbox_inches="tight")
plt.show()